# Is the difference real, and is it big?

**Module 1 · Session 03, part 2**

Rap is more danceable than rock. The gap is about 0.20 on a scale that runs 0 to 1, you can
see it in any chart you draw, and no test is needed to believe it.

Latin is more energetic than pop. The gap there is 0.007.

Both of those come back statistically significant. One of them should change what a curation
team does and the other should change nothing, and the test that produced the verdict cannot
tell you which is which.

## Which test

Three of these turn up today. Choosing between them is the easy part.

| Your question | The test |
|---|---|
| Do two groups differ on a number? | t-test |
| Do three or more groups differ on a number? | one-way ANOVA |
| Are two categorical columns related? | chi-square |
| Do two numbers move together? | correlation, which was part 1 |

They all answer the same narrow question: could this be chance. None of them answers whether
it is big.


In [ ]:
# Setup. Same file as last week, plus one style block so every chart here matches.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import HTML
from scipy import stats

# One theme for the whole notebook, so no chart below needs styling of its own.
# The colours are chosen to stay distinguishable with colour-blindness.
BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#8b8a85"
sns.set_theme(style="whitegrid", palette=[BLUE, ORANGE, GREY],
              rc={"figure.dpi": 110, "grid.color": "#ececea", "font.size": 10,
                  "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left"})

# The file lives in the class repository. pandas reads a URL exactly like a file.
URL = "https://raw.githubusercontent.com/aaubs/ds-master/codex/m1-pandas-2026/data/M1_2026/spotify_songs.csv"

songs = pd.read_csv(URL).rename(columns={
    "track_name": "title", "track_artist": "artist",
    "track_popularity": "popularity", "playlist_genre": "genre"})
tracks = songs.drop_duplicates("track_id")   # one row per song instead of one per placement


def listen(rows, *extra_columns):
    """Show rows with a clickable Spotify link. track_id is a real Spotify ID."""
    out = rows[["title", "artist", *extra_columns]].round(3)
    out["listen"] = "https://open.spotify.com/track/" + rows["track_id"]
    return HTML(out.to_html(render_links=True, escape=False, index=False))


print(f"Rows (a track on a playlist): {len(songs):,}")
print(f"Distinct songs:               {len(tracks):,}")


## Look before you test

Two genres, one column. This uses `songs` rather than `tracks`: a track that sits on both an
edm and a rock playlist counts for both, because the question is about what the playlists hold.


In [ ]:
two = songs[songs["genre"].isin(["edm", "rock"])]
edm = songs.loc[songs["genre"] == "edm", "energy"]
rock = songs.loc[songs["genre"] == "rock", "energy"]

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=two, x="energy", hue="genre", hue_order=["edm", "rock"],
             bins=40, alpha=0.55, palette=[BLUE, ORANGE], ax=ax)
ax.axvline(edm.mean(), color=BLUE, linewidth=2)
ax.axvline(rock.mean(), color=ORANGE, linewidth=2)
ax.set(title="Energy: edm against rock (vertical lines are the means)",
       xlabel="energy", ylabel="rows")
plt.show()

print(f"edm:  {len(edm):,} rows, mean {edm.mean():.3f}")
print(f"rock: {len(rock):,} rows, mean {rock.mean():.3f}")
print(f"gap:  {edm.mean() - rock.mean():.3f}")


Everything between the two lines is the entire finding, and the distributions underneath
overlap heavily. Forty-five percent of the rock rows sit above the edm average.

The ends of the two distributions make the same point louder.


In [ ]:
ends = pd.concat([two[two["genre"] == "rock"].nlargest(1, "energy"),
                  two[two["genre"] == "edm"].nsmallest(1, "energy")])
display(listen(ends, "genre", "energy"))


The most energetic rock track is a 1973 Iggy Pop mix at 0.998, level with the most energetic
edm track in the file. The least energetic edm track is at 0.11. The genre means differ by 0.07
and these two songs differ by 0.9, in the wrong direction.

Write down your answers before running the next cell. Is the gap real? Is it big?

## The t-test

It answers one thing: if these groups really had the same mean, how surprising would a gap
this size be?


In [ ]:
# equal_var=False is Welch's version, which does not assume equal spread. Use it by default.
result = stats.ttest_ind(edm, rock, equal_var=False)
print(f"p value: {result.pvalue:.2e}")


Ten to the minus ninety-seven. If the two genres really had identical mean energy, a gap this
size in samples this size would be so rare the number stops meaning anything you can picture.

Now the same test somewhere the difference is invisible.


In [ ]:
latin = songs.loc[songs["genre"] == "latin", "energy"]
pop = songs.loc[songs["genre"] == "pop", "energy"]

test = stats.ttest_ind(latin, pop, equal_var=False)
print(f"means: {latin.mean():.4f} latin, {pop.mean():.4f} pop, gap {latin.mean() - pop.mean():.4f}")
print(f"p value: {test.pvalue:.4f}")
print("Significant at 5 percent?", test.pvalue < 0.05)


Seven thousandths of a point, on an index bounded at 0 and 1. Nothing anyone could hear.

Significant.

So the word has now been earned by a gap of 0.070 and by a gap of 0.007, and a paper reporting
"p < 0.05" for the second one would be reporting it accurately.

> **Judgement call.** Ask an agent whether two genres differ and you get a correct test, a
> correct p value and a verdict. Whether anybody should care was not in the request and is not
> in the output. On business-sized data that missing half is usually the entire decision.

## Effect size

What separates those two results is how big the gap is relative to how spread out the data
already was. Cohen's d is the usual one for two means.


In [ ]:
def cohens_d(a, b):
    """Difference in means, measured in pooled standard deviations."""
    return (a.mean() - b.mean()) / np.sqrt((a.std() ** 2 + b.std() ** 2) / 2)


print(f"edm against rock:  d = {cohens_d(edm, rock):.2f}")
print(f"latin against pop: d = {cohens_d(latin, pop):.2f}")


The rough labels are 0.2 small, 0.5 medium, 0.8 large. Conventions, not laws. Knowing them
mostly shows you how far below them real results tend to sit.

Put the second result to a decision and it settles itself. Latin energy 0.708, pop 0.701,
significant at 5 percent. What would you change? No playlist gets re-sorted over that. The
test was correct and the answer was worthless.

That gap between "real" and "worth acting on" is where most bad analysis lives.

## Why sample size did all of this

Everything is significant here because each genre has around five thousand rows. Take the
comparison with nothing in it and run it on thirty rows a side instead, two hundred times.


In [ ]:
p_values = np.array([
    stats.ttest_ind(pop.sample(30, random_state=seed),
                    latin.sample(30, random_state=seed + 999), equal_var=False).pvalue
    for seed in range(200)
])

fig, ax = plt.subplots(figsize=(8, 3.4))
sns.histplot(x=p_values, bins=20, color=BLUE, ax=ax)
ax.axvline(0.05, color=ORANGE, linewidth=2)
ax.text(0.062, ax.get_ylim()[1] * 0.85, "0.05", color=ORANGE)
ax.set(title="p values from 200 runs on 30 rows a side", xlabel="p value", ylabel="runs")
plt.show()

print(f"median p: {np.median(p_values):.3f}")
print(f"significant in {(p_values < 0.05).mean():.1%} of runs")


About five percent of the runs land left of the orange line, which is what you get from chance alone
when there is nothing to find.

The data never changed. Only the row count did. A p value is a statement about a difference
*and* your sample size, and with enough rows any difference that is not exactly zero will cross
any threshold you choose.

## Three or more groups


In [ ]:
by_genre = songs.groupby("genre")["energy"]
anova = stats.f_oneway(*[group for _, group in by_genre])
print(f"F: {anova.statistic:.0f}, p: {anova.pvalue:.1e}")

grand_mean = songs["energy"].mean()
between = (by_genre.size() * (by_genre.mean() - grand_mean) ** 2).sum()
total = ((songs["energy"] - grand_mean) ** 2).sum()
print(f"eta squared: {between / total:.3f}")


Genre accounts for about 14 percent of the variation in energy. The other 86 percent sits
inside genres: two edm tracks usually differ from each other by more than the average edm track
differs from the average rock track.

One sentence, and it describes the dataset better than any p value in this notebook.

## Two categorical columns

`mode` is 1 for major and 0 for minor.


In [ ]:
table = pd.crosstab(songs["genre"], songs["mode"])
table.columns = ["minor", "major"]
shares = table.div(table.sum(axis=1), axis=0)

minor = shares["minor"].sort_values()

fig, ax = plt.subplots(figsize=(7, 3.2))
sns.barplot(x=minor.values, y=minor.index, color=BLUE, ax=ax)
ax.bar_label(ax.containers[0], fmt="{:.0%}".format, padding=4)
ax.set(title="Share of tracks in a minor key", xlabel="", ylabel="", xlim=(0, 0.6))
ax.set_xticks([])
sns.despine(bottom=True)
plt.show()


In [ ]:
chi2, p_value, dof, expected = stats.chi2_contingency(table)
cramers_v = np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1)))
print(f"chi-square: {chi2:.0f}, p: {p_value:.1e}")
print(f"Cramer's V: {cramers_v:.3f}")


Significant, with a Cramer's V of 0.13, which is weak.

The chart says something the V does not. Rock sits at 30 percent minor and everything else
between 41 and 48, so rock is the story and the other five genres bunch together and drag the
single summary number down. A six-by-two table compressed into one statistic loses that.

## Before you go hunting

Run twenty tests on data with nothing in it and about one comes back significant. That is what
a 5 percent threshold means: the rate at which you expect to be fooled.

It used to be a slow problem, because twenty tests took an afternoon. An agent will run fifty
comparisons in one call and hand back the three that cleared 0.05, which is a machine for
manufacturing findings that are not there.

Test the question you arrived with. If you do go fishing across every pair of columns, say so,
and treat whatever surfaces as something to check on other data rather than as a result.

## How to report it

Three things, in this order, and never the third on its own:

1. the size of the difference, in the units you measured
2. how many observations it rests on
3. the p value

"Mean energy is 0.80 for edm and 0.73 for rock, across 6,043 and 4,951 rows, a gap of 0.07 and
d = 0.41 (p < 0.001)" is a sentence somebody can argue with. "The difference was significant"
is not.

## Your turn

Pick a comparison in this file you would actually want the answer to. Any two groups, any
column. Report the counts, the size of the gap, the effect size, then the p value.

Then write the one sentence you would send a curation team, and one saying what it does not
tell them.


In [ ]:
# Your comparison.


## One answer: danceability, rap against rock


In [ ]:
rap_d = songs.loc[songs["genre"] == "rap", "danceability"]
rock_d = songs.loc[songs["genre"] == "rock", "danceability"]

print(f"rap:  {len(rap_d):,} rows, mean {rap_d.mean():.3f}")
print(f"rock: {len(rock_d):,} rows, mean {rock_d.mean():.3f}")
print(f"gap: {rap_d.mean() - rock_d.mean():.3f}, d = {cohens_d(rap_d, rock_d):.2f}")
print(f"p value: {stats.ttest_ind(rap_d, rock_d, equal_var=False).pvalue:.1e}")


An effect size of 1.4, which is enormous by any convention, and one of the largest gaps
between any two genres in the file. A curation team should care.

Notice this is also where the p value adds least. Nothing about the conclusion depends on it,
and you could have reached the same answer from the chart.

## Takeaways

- A p value answers whether a difference exists given your sample size. It says nothing about
  size.
- On business-sized tables nearly everything is significant, so effect size carries the finding.
- Report size, then counts, then the p value.
- Genre explains about 14 percent of the variation in energy. Most of it sits within genres.
- Twenty tests on nothing produce roughly one significant result.
- All of this describes one 2020 playlist snapshot, and six playlist genres are a narrow slice of
  what the world listens to.
